In [1]:
"""
Bank Customer Churn Predictor
This script processes bank customer data, engineers new financial features, 
builds a Logistic Regression pipeline with balanced class weights, and evaluates 
the model using ROC-AUC and custom decision thresholds.
"""

import sys
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
# ==========================================
# 1. DATA LOADING & EXPLORATION
# ==========================================
# Safely load the dataset (Using a relative path for GitHub portability)
FILE_PATH = 'Bank_Churn.csv'

try:
    df = pd.read_csv(FILE_PATH)
except FileNotFoundError:
    print(f"Error: Dataset '{FILE_PATH}' not found. Please ensure it is in the same directory.")
    sys.exit(1)

# Drop columns that hold no predictive value (Identifiable Information)
columns_to_drop = ['CustomerId', 'Surname']
df.drop(columns=columns_to_drop, axis=1, inplace=True)

# Display basic data quality metrics
max_nulls = df.isnull().sum().max()
print(f"--- Data Quality Check ---")
print(f"Maximum Null Values in any column: {max_nulls}")

# Display the class imbalance ratio
churn_percentage = df['Exited'].value_counts(normalize=True) * 100
print(f"\n--- Class Distribution (Churn Percentage) ---\n{churn_percentage.to_string()}\n")

--- Data Quality Check ---
Maximum Null Values in any column: 0

--- Class Distribution (Churn Percentage) ---
Exited
0    79.63
1    20.37



In [3]:
# ==========================================
# 2. DATA SPLITTING & FEATURE ENGINEERING
# ==========================================
# Define features (X) and target (y)
X = df.drop(columns=['Exited'])
y = df['Exited']

# Stratified split ensures the train and test sets have the same proportion of churn cases
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

def apply_feature_engineering(data: pd.DataFrame) -> pd.DataFrame:
    """
    Engineers new business-logic features to improve model performance:
    - tenure_to_age: Ratio of how long they've been with the bank relative to their age.
    - balance_salary_ratio: Ratio of their account balance to their estimated salary.
    """
    data = data.copy()
    data['tenure_to_age'] = data['Tenure'] / data['Age']
    data['balance_salary_ratio'] = data['Balance'] / data['EstimatedSalary']
    return data

# Apply feature engineering strictly to train and test sets independently
X_train = apply_feature_engineering(X_train)
X_test = apply_feature_engineering(X_test)

In [4]:
# ==========================================
# 3. PIPELINE CONFIGURATION
# ==========================================
# Categorize columns based on their data types (including newly engineered features)
numeric_columns =[
    'CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 
    'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 
    'tenure_to_age', 'balance_salary_ratio'
]
categorical_columns = ['Geography', 'Gender']

# Set up the preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first'), categorical_columns)
    ]
)

# Build a unified pipeline combining the preprocessor and the Logistic Regression model
# Note: class_weight='balanced' is used to penalize mistakes on the minority class (Churn)
pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(class_weight='balanced', random_state=42))
    ]
)

print("Training the Logistic Regression pipeline...\n")
pipeline.fit(X_train, y_train)

Training the Logistic Regression pipeline...



,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [5]:
# ==========================================
# 4. MODEL EVALUATION
# ==========================================
# Generate standard predictions and probabilities
y_pred_standard = pipeline.predict(X_test)
churn_probabilities = pipeline.predict_proba(X_test)[:, 1]

# Calculate ROC-AUC Score
roc_auc = roc_auc_score(y_test, churn_probabilities)

print("="*45)
print("  EVALUATION 1: STANDARD THRESHOLD (0.50)")
print("="*45)
print(f"ROC-AUC Score: {roc_auc:.4f}\n")
print(classification_report(y_test, y_pred_standard))

  EVALUATION 1: STANDARD THRESHOLD (0.50)
ROC-AUC Score: 0.7754

              precision    recall  f1-score   support

           0       0.91      0.71      0.80      1593
           1       0.39      0.71      0.50       407

    accuracy                           0.71      2000
   macro avg       0.65      0.71      0.65      2000
weighted avg       0.80      0.71      0.74      2000



In [6]:
# ==========================================
# 5. CUSTOM THRESHOLD TUNING
# ==========================================
print("="*45)
print("  EVALUATION 2: STRICT THRESHOLD (0.65)")
print("="*45)
# Apply a custom threshold to increase Precision (reduce false positive churn alerts)
CUSTOM_THRESHOLD = 0.65
y_pred_custom = (churn_probabilities >= CUSTOM_THRESHOLD).astype(int)

print(f"Classification Report (Threshold = {CUSTOM_THRESHOLD}):\n")
print(classification_report(y_test, y_pred_custom))

  EVALUATION 2: STRICT THRESHOLD (0.65)
Classification Report (Threshold = 0.65):

              precision    recall  f1-score   support

           0       0.87      0.87      0.87      1593
           1       0.49      0.47      0.48       407

    accuracy                           0.79      2000
   macro avg       0.68      0.67      0.68      2000
weighted avg       0.79      0.79      0.79      2000

